# Fase 2. Modelos de demanda

Este notebook revisa los resultados de `make train`. La lógica de entrenamiento, ajuste y registro está en `mibici/modeling`; aquí solo se consultan MLflow y el modelo registrado.

In [1]:
import mlflow
import pandas as pd

from mibici.config import (
    MLFLOW_MODEL_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    PANEL_FILE,
    REGISTERED_MODEL_NAME,
    TEST_START,
)
from mibici.features import MODEL_FEATURES, TARGET
from mibici.modeling.evaluation import metrics_by_group, regression_metrics, temporal_split
from mibici.modeling.predict import FORECAST_COLUMN, DemandForecaster

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## Partición temporal

In [2]:
panel = pd.read_parquet(PANEL_FILE)
train, test = temporal_split(panel, TEST_START)
pd.DataFrame(
    {
        'filas': [len(train), len(test)],
        'desde': [train['hour_ts'].min(), test['hour_ts'].min()],
        'hasta': [train['hour_ts'].max(), test['hour_ts'].max()],
        'pct_horas_sin_viajes': [
            round((train[TARGET] == 0).mean() * 100, 1),
            round((test[TARGET] == 0).mean() * 100, 1),
        ],
    },
    index=['entrenamiento', 'prueba'],
)

,filas,desde,hasta,pct_horas_sin_viajes
entrenamiento,799176,2026-07-08,2026-09-16 23:00:00,59.0
prueba,157584,2026-09-17,2026-09-30 23:00:00,50.7


## Comparación de modelos en MLflow

Un run por modelo y por línea base, con sus métricas en el periodo de prueba.

In [3]:
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_MODEL_EXPERIMENT], order_by=['attributes.start_time DESC']
)
runs = runs[runs['tags.tipo'].isin(['modelo', 'linea_base'])]
cols = ['tags.modelo', 'tags.tipo', 'metrics.cv_rmse', 'metrics.test_MAE', 'metrics.test_RMSE',
        'metrics.test_WAPE_pct', 'metrics.test_sesgo']
runs = runs.drop_duplicates('tags.modelo')  # la corrida más reciente de cada modelo
runs[cols].sort_values('metrics.test_MAE').round(3)

,tags.modelo,tags.tipo,metrics.cv_rmse,metrics.test_MAE,metrics.test_RMSE,metrics.test_WAPE_pct,metrics.test_sesgo
8,gradient_boosting,modelo,1.271,0.862,1.502,54.539,-0.133
22,ridge,modelo,1.387,0.949,1.615,60.012,-0.067
24,ingenuo_lag_168h,linea_base,NaN,1.037,1.936,65.594,-0.159
15,poisson_glm,modelo,1.513,1.037,1.893,65.613,-0.075
23,media_movil_24h,linea_base,NaN,1.421,2.325,89.890,-0.039


## Ajuste de hiperparámetros

Cada combinación probada es un run anidado. Se muestran las mejores por modelo.

In [4]:
trials = mlflow.search_runs(
    experiment_names=[MLFLOW_MODEL_EXPERIMENT],
    filter_string="tags.tipo = 'combinacion'",
    order_by=['metrics.cv_rmse ASC'],
)
param_cols = [c for c in trials.columns if c.startswith('params.')]
trials.groupby('tags.modelo').head(3)[['tags.modelo', 'metrics.cv_rmse', 'metrics.cv_mae', *param_cols]]

,tags.modelo,metrics.cv_rmse,metrics.cv_mae,params.max_leaf_nodes,params.max_iter,params.l2_regularization,params.min_samples_leaf,params.loss,params.learning_rate,params.alpha
0,gradient_boosting,1.271112,0.703138,31,200,0.0,100,squared_error,0.2,NaN
1,gradient_boosting,1.272684,0.701338,63,300,1.0,20,squared_error,0.2,NaN
2,gradient_boosting,1.277005,0.709333,31,200,1.0,20,squared_error,0.05,NaN
8,ridge,1.386670,0.758568,NaN,NaN,NaN,NaN,NaN,NaN,0.01
9,ridge,1.386670,0.758568,NaN,NaN,NaN,NaN,NaN,NaN,0.1
10,ridge,1.386670,0.758569,NaN,NaN,NaN,NaN,NaN,NaN,1.0
14,poisson_glm,1.512653,0.827426,NaN,NaN,NaN,NaN,NaN,NaN,0.1
15,poisson_glm,1.525713,0.823941,NaN,NaN,NaN,NaN,NaN,NaN,1e-06
16,poisson_glm,1.525731,0.823949,NaN,NaN,NaN,NaN,NaN,NaN,1e-05


## Modelo registrado

Versiones en el Model Registry y la que tiene el alias `champion`.

In [5]:
client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
pd.DataFrame(
    [
        {
            'version': int(v.version),
            'modelo': v.tags.get('modelo'),
            'cv_rmse': v.tags.get('cv_rmse'),
            'test_MAE': v.tags.get('test_MAE'),
            'hiperparametros': v.tags.get('hiperparametros'),
        }
        for v in versions
    ]
).sort_values('version')

,version,modelo,cv_rmse,test_MAE,hiperparametros
2,1,gradient_boosting,1.271112,0.862083,"{""min_samples_leaf"": 100, ""max_leaf_nodes"": 31..."
1,2,ridge,1.386670,0.948593,"{""alpha"": 0.01}"
0,3,poisson_glm,1.512653,1.037123,"{""alpha"": 0.1}"


In [6]:
forecaster = DemandForecaster()
forecast = forecaster.predict(test)
pd.Series(regression_metrics(forecast[TARGET], forecast[FORECAST_COLUMN])).round(3).to_frame('champion')

,champion
MAE,0.862
RMSE,1.502
WAPE_pct,54.539
sesgo,-0.133


## Error por grupo de estación y por hora

In [7]:
metrics_by_group(forecast[TARGET], forecast[FORECAST_COLUMN], test['station_cluster']).round(3)

,grupo,filas,demanda_media,MAE,RMSE,WAPE_pct,sesgo
0,-1,1344,0.390,0.364,0.739,93.366,-0.003
1,0,63168,1.394,0.773,1.353,55.432,-0.118
2,1,71904,1.966,1.015,1.693,51.604,-0.167
3,2,21168,0.903,0.642,1.248,71.061,-0.073


In [8]:
metrics_by_group(forecast[TARGET], forecast[FORECAST_COLUMN], test['hour'], 'hora').round(3)

,hora,filas,demanda_media,MAE,RMSE,WAPE_pct,sesgo
0,0,6566,0.242,0.309,0.609,127.676,-0.026
1,1,6566,0.001,0.023,0.076,3046.315,0.022
2,2,6566,0.000,0.008,0.015,NaN,0.008
3,3,6566,0.000,0.010,0.017,NaN,0.010
4,4,6566,0.000,0.028,0.038,NaN,0.028
5,5,6566,0.328,0.311,0.543,94.692,-0.042
6,6,6566,1.178,0.731,1.098,62.025,-0.128
7,7,6566,2.246,1.075,1.637,47.881,-0.243
8,8,6566,2.839,1.300,1.906,45.794,-0.300
9,9,6566,2.031,1.122,1.642,55.265,-0.108


## Demanda diaria del sistema: real contra pronóstico

La suma de todas las estaciones por día muestra si el modelo subestima o sobrestima.

In [9]:
daily = forecast.groupby(forecast['hour_ts'].dt.date)[[TARGET, FORECAST_COLUMN]].sum()
daily['diferencia_pct'] = (daily[FORECAST_COLUMN] / daily[TARGET] - 1) * 100
daily.round(1)

,departures,pronostico,diferencia_pct
hour_ts,,,
2026-09-17,18858,13873.7,-26.4
2026-09-18,19669,18162.4,-7.7
2026-09-19,13612,12671.1,-6.9
2026-09-20,12387,12104.7,-2.3
2026-09-21,19275,16984.6,-11.9
2026-09-22,20263,19076.5,-5.9
2026-09-23,19133,17195.2,-10.1
2026-09-24,18729,17658.9,-5.7
2026-09-25,19367,18305.2,-5.5


## Por qué la regresión Poisson no supera a Ridge

Se carga la versión de la regresión Poisson y se revisan sus mayores errores.

In [10]:
latest = {v.tags['modelo']: v.version for v in sorted(versions, key=lambda v: int(v.version))}
poisson = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{latest['poisson_glm']}")
pred = pd.Series(poisson.predict(test[MODEL_FEATURES]), index=test.index)
sq_error = (pred - test[TARGET]) ** 2
worst = sq_error.nlargest(int(len(sq_error) * 0.001)).index
pd.Series(
    {
        'pronostico_maximo': pred.max(),
        'real_maximo': test[TARGET].max(),
        'pct_error_cuadratico_del_0.1_pct_de_filas': sq_error[worst].sum() / sq_error.sum() * 100,
    }
).round(1)

pronostico_maximo                            81.3
real_maximo                                  65.0
pct_error_cuadratico_del_0.1_pct_de_filas    18.2
dtype: float64

In [11]:
cols = ['station_id', 'hour_ts', TARGET, 'dep_lag_1', 'dep_lag_24', 'dep_lag_168', 'dep_roll_mean_24']
test.loc[sq_error.nlargest(5).index, cols].assign(pronostico_poisson=pred.round(1))

,station_id,hour_ts,departures,dep_lag_1,dep_lag_24,dep_lag_168,dep_roll_mean_24,pronostico_poisson
65272,35,2026-09-30 16:00:00,20,34.0,23.0,32.0,14.958333,67.4
97838,51,2026-09-27 14:00:00,26,32.0,41.0,39.0,18.125000,72.9
358986,194,2026-09-28 18:00:00,56,15.0,1.0,28.0,3.041667,10.6
64984,35,2026-09-18 16:00:00,19,35.0,29.0,25.0,14.500000,64.4
65151,35,2026-09-25 15:00:00,36,32.0,34.0,35.0,14.666667,81.3
